# Análisis del Observatorio Analítico del Turismo en Colombia

Este notebook tiene como objetivo la construcción de una base de datos maestra para el análisis del sector turístico en Colombia, integrando información de diversas fuentes oficiales como Migración Colombia, DANE, Banco de la República y el Ministerio de Comercio, Industria y Turismo (MinCIT).

## 1. Identificación de Variables Claves Extraídas
Las siguientes son las variables clave que han sido extraídas y procesadas en este análisis, basándose en la disponibilidad de datos de la fuente del MinCIT y otras APIs:

### Migración Colombia (Datos de Extranjeros No Residentes)
*   **Destino por Región:** Número de viajeros a las regiones de Amazonía, Andina, Caribe, Insular, Orinoquía, Pacífica.
*   **Motivo de Viaje:** Desagregación por Convenciones y eventos, Educación y formación, Negocios y motivos profesionales, Otros, Salud y atención médica, Tránsito, Vacaciones, recreo y ocio, Visitas a familiares y amigos.
*   **Sexo:** Número de viajeros según sexo (Femenino, Masculino, Otro).
*   **Total de Visitantes No Residentes:** Cifras totales de visitantes no residentes, turistas, excursionistas, viajeros en tránsito y tripulantes.

### DANE (Población Ocupada y PIB)
*   **Población Ocupada en el Sector Turístico:** Ocupados totales nacionales, y desagregados por ramas de actividad como Comercio y reparación de vehículos, Alojamiento y servicios de comida, Transporte y almacenamiento.
*   **PIB Alojamiento y Servicios de Comida:** Valor del PIB del sector, su variación porcentual anual y su participación en el PIB total.

### Banco de la República (Tasa Representativa del Mercado - TRM)
*   **TRM:** Promedio trimestral y valor de cierre trimestral.

### Datos.gov.co (Visitantes Extranjeros por Continente)
*   **Origen por Continente:** Número de visitantes extranjeros de América, Europa, Asia, África, Oceanía y Otros continentes.

Estas variables se utilizan en conjunto con el Año y el Trimestre para construir la base de datos maestra.

## 2. Selección de la Fuente de Datos
Para optimizar la recopilación y asegurar la coherencia de los datos, se seleccionó una única fuente principal: un archivo de Excel consolidado proveniente del **Ministerio de Comercio, Industria y Turismo (MinCIT)**. Esta fuente actúa como un compilado de datos de las otras entidades, enfocándose específicamente en indicadores turísticos, lo que facilita la integración y reduce la redundancia.

## 3. Montaje de la Base de Datos en GitHub
El archivo fuente original se ha versionado y hecho público a través de un repositorio en GitHub. Esto permite la trazabilidad, el acceso colaborativo y la facilidad de descarga directa desde el entorno de Colab.

A continuación, se procede a clonar el repositorio para acceder a los datos.

## 4. Establecimiento de la Periodicidad de Análisis
Después de una revisión exhaustiva de las bases de datos originales, se determinó que la **periodicidad trimestral** es la más consistente y conveniente para todas las variables identificadas. Esta elección permite un análisis robusto de tendencias y estacionalidades, equilibrando la granularidad de los datos con la disponibilidad y uniformidad entre las diferentes fuentes.



## 5. Extracción y Limpieza de Datos por Sección
La extracción de datos se ha realizado de forma modular, dividiendo el proceso por las hojas de cálculo correspondientes en el archivo fuente del MinCIT. Cada sección sigue un proceso como este:
1.  **Carga de las Hojas:** Importación de la hoja específica desde el archivo Excel.
2.  **Limpieza Inicial:** Manejo de filas no útiles (notas, fuentes), estandarización de nombres de columnas.
3.  **Ajuste de Periodicidad:** Conversión de datos mensuales a trimestrales cuando sea necesario.
4.  **Agrupación y Agregación:** Consolidación de datos a nivel trimestral y por las dimensiones relevantes (ej. sexo, motivo, región, etc.).
5.  **Filtro por Rango Temporal:** Selección de los años de interés, en este caso, desde **2015** hasta **2024**, para asegurar la coherencia temporal de la base maestra.

Para cada una de estas secciones, se han desarrollado funciones adaptadas a las particularidades de cada hoja, garantizando la calidad y consistencia de los datos resultantes.

In [1]:
!git clone https://github.com/hivalerieee/Observatorio-analitico-del-turismo-en-colombia.git

Cloning into 'Observatorio-analitico-del-turismo-en-colombia'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 8.06 MiB | 19.19 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:
import pandas as pd

### 5.1. Datos de Migración - Extranjeros No Residentes
Esta sección se enfoca en la extracción y procesamiento de datos relacionados con los visitantes extranjeros no residentes, incluyendo su ciudad de destino, motivo de viaje y desagregación por sexo. Estos datos provienen de las hojas "Extranjeros Ciudad Destino", "Extranjeros Motivo viaje" y "Extranjeros por Sexo" del archivo Excel.

Se utiliza una función `limpiar_hoja` para estandarizar la carga, filtrar años, mapear meses a trimestres y preparar los DataFrames para su consolidación. Luego, los datos se agrupan a nivel trimestral y se pivotean las categorías relevantes (sexo, motivo, región) para facilitar la unión posterior.

Además se hace un ajuste ya que los datos iniciales vienen por ciudad lo cual hace complicado pivotear entre ellas, por lo que se modifica para que se observe por región.

In [3]:
#Datos de migracion y llegada de extranjeros no residentes
ruta_excel = '/content/Observatorio-analitico-del-turismo-en-colombia/26-01-2025-OEE-EC-ESTADISTICAS-TURISMO-MINCIT.xlsx'

#Nombres exactos de las hojas a extraer y la fila donde inicia cada columna
df_ciudad = pd.read_excel(ruta_excel, sheet_name="Extranjeros Ciudad Destino", header=8)
df_motivo = pd.read_excel(ruta_excel, sheet_name="Extranjeros Motivo viaje", header=7)
df_sexo   = pd.read_excel(ruta_excel, sheet_name="Extranjeros por Sexo", header=8)



In [4]:
import pandas as pd
ruta_excel = '/content/Observatorio-analitico-del-turismo-en-colombia/26-01-2025-OEE-EC-ESTADISTICAS-TURISMO-MINCIT.xlsx'
meses_map = {
    "enero":1, "febrero":2, "marzo":3, "abril":4, "mayo":5, "junio":6,
    "julio":7, "agosto":8, "septiembre":9, "octubre":10, "noviembre":11, "diciembre":12
}

def limpiar_hoja(sheet_name, header_row, usecols):
    df = pd.read_excel(ruta_excel, sheet_name=sheet_name, header=header_row, usecols=usecols)
    df = df[pd.to_numeric(df["Año"], errors="coerce").notna()].copy()
    df["Año"] = df["Año"].astype(int)
    df["Mes"] = df["Mes"].astype(str).str.strip().str.lower()
    df["mes_num"] = df["Mes"].map(meses_map)
    df["Trimestre"] = ((df["mes_num"] - 1) // 3) + 1
    return df

 #Cargar y limpiar cada hoja
df_ciudad = limpiar_hoja("Extranjeros Ciudad Destino", header_row=8, usecols="A:G")
df_motivo = limpiar_hoja("Extranjeros Motivo viaje",    header_row=7, usecols="A:D")
df_sexo   = limpiar_hoja("Extranjeros por Sexo",        header_row=8, usecols="A:D")

#Agregar a nivel Año-Trimestre
sexo_q = (df_sexo.groupby(["Año","Trimestre","Sexo"])["Viajeros"].sum()
          .unstack("Sexo").add_prefix("sexo_").reset_index())

motivo_q = (df_motivo.groupby(["Año","Trimestre","Motivo de Viaje"])["Viajeros"].sum()
            .unstack("Motivo de Viaje").add_prefix("motivo_").reset_index())

ciudad_q = (df_ciudad.groupby(["Año","Trimestre"])["Viajeros"].sum()
            .reset_index().rename(columns={"Viajeros":"viajeros_total_ciudad"}))

# Mapeo departamento -> región
region_map = {
    # Región Caribe
    "Atlántico": "Caribe", "Bolívar": "Caribe", "Cesar": "Caribe",
    "Córdoba": "Caribe", "La Guajira": "Caribe", "Magdalena": "Caribe",
    "Sucre": "Caribe",
    # Región Insular
    "San Andrés y Providencia": "Insular",
    # Región Pacífica
    "Chocó": "Pacífica", "Valle del Cauca": "Pacífica",
    "Cauca": "Pacífica", "Nariño": "Pacífica",
    # Región Andina
    "Antioquia": "Andina", "Boyacá": "Andina", "Caldas": "Andina",
    "Cundinamarca": "Andina", "Bogotá, D.C.": "Andina", "Huila": "Andina",
    "Norte de Santander": "Andina", "Quindío": "Andina", "Risaralda": "Andina",
    "Santander": "Andina", "Tolima": "Andina",
    # Región Orinoquía
    "Arauca": "Orinoquía", "Casanare": "Orinoquía",
    "Meta": "Orinoquía", "Vichada": "Orinoquía",
    # Región Amazonía
    "Amazonas": "Amazonía", "Caquetá": "Amazonía", "Guainía": "Amazonía",
    "Guaviare": "Amazonía", "Putumayo": "Amazonía", "Vaupés": "Amazonía",
    # Sin especificar
    "Sin Especificar": "Sin Especificar",
}

df_ciudad["Region"] = df_ciudad["Departamento"].map(region_map)

sin_mapear = df_ciudad[df_ciudad["Region"].isna()]["Departamento"].unique()

region_q = (df_ciudad.groupby(["Año","Trimestre","Region"])["Viajeros"].sum()
            .unstack("Region").add_prefix("region_").reset_index())
#Unir las 3 en un solo dataframe por Año + Trimestre
df_final = region_q.merge(motivo_q, on=["Año","Trimestre"], how="outer") \
                    .merge(sexo_q, on=["Año","Trimestre"], how="outer")

df_final = df_final.sort_values(["Año","Trimestre"]).reset_index(drop=True)
print(df_final.shape)
df_final.head(10)

df_final = df_final[df_final['Año'].between(2015, 2024)]
pd.concat([df_final.head(10), df_final.tail()])

df_sexo_motivo_ciudad = df_final.copy()


(56, 20)


### 5.2. Datos de Población Ocupada en Sectores Turísticos
Aquí se procesa la información sobre la población ocupada en ramas de actividad económica directamente relacionadas con el turismo, como "Alojamiento y servicios de comida", "Comercio y reparación de vehículos" y "Transporte y almacenamiento". Los datos se extraen de la hoja "Ocupados Mensual GEIH" del archivo fuente.

El proceso incluye la limpieza de años, la conversión de meses a trimestres y la agregación de los valores de ocupados a un nivel trimestral, calculando el promedio mensual dentro de cada trimestre.

In [5]:
#Población ocupada según ramas de actividad CIIU 4 A.C.
import pandas as pd

df_Poblacion_Ocupada = pd.read_excel(ruta_excel, sheet_name="Ocupados Mensual GEIH", skiprows=7, nrows = 138) #esta mensual

# Asegurar que la columna 'Año' sea de tipo entero
df_Poblacion_Ocupada['Año'] = pd.to_numeric(df_Poblacion_Ocupada['Año'], errors='coerce')
df_Poblacion_Ocupada.dropna(subset=['Año'], inplace=True)
df_Poblacion_Ocupada['Año'] = df_Poblacion_Ocupada['Año'].astype(int)

#Diccionario para mapear los nombres abreviados de los meses
meses_a_trimestre = {
    'Ene': '1', 'Feb': '1', 'Mar': '1',
    'Abr': '2', 'May': '2', 'Jun': '2',
    'Jul': '3', 'Ago': '3', 'Sep': '3',
    'Oct': '4', 'Nov': '4', 'Dic': '4'
}

# Estandarizar la columna 'Mes' para que coincida con las claves del diccionario
df_Poblacion_Ocupada['Mes'] = df_Poblacion_Ocupada['Mes'].astype(str).str.strip().str[:3].str.capitalize()

#  Crear la columna Trimestre
df_Poblacion_Ocupada['Trimestre'] = df_Poblacion_Ocupada['Mes'].map(meses_a_trimestre)

df_Poblacion_Ocupada['Trimestre'] = df_Poblacion_Ocupada['Trimestre'].fillna(0).astype(int)

# Columnas numéricas a agrupar
cols_num = [
    'Ocupados Total Nacional',
    'Comercio y reparación de vehículos',
    'Alojamiento y servicios de comida',
    'Transporte y almacenamiento'
]

#  Agrupar por Año y Trimestre (sacando el promedio de los meses del trimestre)
df_Poblacion_Ocupada_trimestral = (
    df_Poblacion_Ocupada
    .groupby(['Año', 'Trimestre'])[cols_num]
    .mean()
    .reset_index()
)

df_Poblacion_Ocupada_trimestral = df_Poblacion_Ocupada_trimestral[df_Poblacion_Ocupada_trimestral['Año'].between(2015, 2024)]
pd.concat([df_Poblacion_Ocupada_trimestral.head(), df_Poblacion_Ocupada_trimestral.tail()])

df_poblacion_ocupada = df_Poblacion_Ocupada_trimestral.copy()


### 5.3. Tasa Representativa del Mercado (TRM)
Esta sección se encarga de obtener la Tasa Representativa del Mercado (TRM) histórica, que es un factor crucial para el análisis económico del turismo. Los datos de la TRM se cargan directamente desde un archivo CSV alojado en el mismo repositorio de GitHub.

La función `limpiar_trm_github` realiza la limpieza de valores monetarios, la conversión de fechas, la extracción de año y trimestre, y la agregación para obtener el promedio y el valor de cierre de la TRM por trimestre.

In [6]:
#TRM
import pandas as pd

# URL de la versión RAW del archivo en GitHub
url_trm = "https://raw.githubusercontent.com/hivalerieee/Observatorio-analitico-del-turismo-en-colombia/main/TRM.csv"

def limpiar_trm_github(url, anio_inicio=2015, anio_fin=2024):
    # Cargar directamente el CSV desde GitHub
    df = pd.read_csv(url)

    # Limpiar el formato del valor numérico
    df["VALOR_NUM"] = (
        df["VALOR"]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

    # Convertir la fecha a formato datetime y extraer Año y Trimestre
    df["FECHA"] = pd.to_datetime(df["VIGENCIADESDE"], format="%d/%m/%Y")
    df["Año"] = df["FECHA"].dt.year.astype(int)
    df["Trimestre"] = df["FECHA"].dt.quarter.astype(int)

    # Filtar del 2015 en adelante
    df = df[(df["Año"] >= anio_inicio) & (df["Año"] <= anio_fin)].copy()

    # Agrupar por Año y Trimestre
    trm_q = df.groupby(["Año", "Trimestre"]).agg(
        TRM_promedio=("VALOR_NUM", "mean"),
        TRM_cierre=("VALOR_NUM", "last")
    ).reset_index()

    # Redondear valores
    trm_q["TRM_promedio"] = trm_q["TRM_promedio"].round(2)
    trm_q["TRM_cierre"] = trm_q["TRM_cierre"].round(2)

    # Ordenar cronológicamente
    trm_q = trm_q.sort_values(["Año", "Trimestre"]).reset_index(drop=True)

    return trm_q

#Cargar y procesar la TRM desde 2015 y hasta 2024
df_trm = limpiar_trm_github(url_trm, anio_inicio=2015, anio_fin=2024)

print(df_trm.shape)

# Arreglar el formato de visualizacion
df_trm.head(10).style.format({
    "TRM_promedio": lambda x: f"${x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'),
    "TRM_cierre": lambda x: f"${x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
})

Trm_Trimestral = df_trm.copy()

(40, 4)


### 5.4. Visitas de Extranjeros No Residentes (Total)
Esta parte del proceso se dedica a extraer y consolidar los datos generales de visitantes no residentes. La información proviene de la hoja "Visitantes No Residentes" del archivo Excel.

La función `limpiar_visitantes_no_residentes` estandariza el proceso de carga, asegurando que los años estén en el rango deseado, los meses se mapeen a trimestres y las columnas métricas sean numéricas. Finalmente, los totales de visitantes se suman por trimestre.

In [7]:
# Visitantes no residentes
import pandas as pd

meses_map = {
    "enero":1, "febrero":2, "marzo":3, "abril":4, "mayo":5, "junio":6,
    "julio":7, "agosto":8, "septiembre":9, "octubre":10, "noviembre":11, "diciembre":12
}

def limpiar_visitantes_no_residentes(header_row=8, usecols="A:G", anio_inicio=2015, anio_fin=2024):
    #Cargar la hoja desde la fila 9 de Excel (header=8)
    df = pd.read_excel(
        ruta_excel,
        sheet_name="Visitantes No Residentes",
        header=header_row,
        usecols=usecols
    )

    #Limpiar espacios en los nombres de las columnas
    df.columns = df.columns.astype(str).str.strip()

    # Filtrar filas donde 'Año' no sea numérico
    df = df[pd.to_numeric(df["Año"], errors="coerce").notna()].copy()
    df["Año"] = df["Año"].astype(int)
    df = df[(df["Año"] >= anio_inicio) & (df["Año"] <= anio_fin)].copy()

    # Limpiar mes y calcular trimestre
    df["Mes"] = df["Mes"].astype(str).str.strip().str.lower()
    df["mes_num"] = df["Mes"].map(meses_map)
    df["Trimestre"] = ((df["mes_num"] - 1) // 3) + 1

    # Descartar filas con mes no válido
    df = df.dropna(subset=["Trimestre"]).copy()
    df["Trimestre"] = df["Trimestre"].astype(int)

    # Asegurar que las columnas métricas sean numéricas
    columnas_metricas = [col for col in df.columns if col not in ["Año", "Mes", "mes_num", "Trimestre"]]

    for col in columnas_metricas:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Agregar a nivel Año-Trimestre (sumando todos los meses del trimestre)
    visitantes_q = df.groupby(["Año", "Trimestre"])[columnas_metricas].sum().reset_index()

    # Ordenar cronológicamente
    visitantes_q = visitantes_q.sort_values(["Año", "Trimestre"]).reset_index(drop=True)

    return visitantes_q

# Cargar la hoja de Visitantes No Residentes
df_visitantes = limpiar_visitantes_no_residentes(anio_inicio=2015, anio_fin=2024)

# Mostrar resultado
print(df_visitantes.shape)

#separador de miles por punto
df_visitantes.head(10).style.format({
    col: lambda x: f"{x:,.0f}".replace(',', '.') if pd.notna(x) else '-'
    for col in df_visitantes.columns if col not in ["Año", "Trimestre"]
})

df_visitantes_no_residentes = df_visitantes.copy()

(40, 7)


### 5.5. Producto Interno Bruto (PIB) del Sector Alojamiento y Servicios de Comida
Se extraen los datos del Producto Interno Bruto (PIB) específicamente para las actividades de alojamiento y servicios de comida, un indicador clave de la contribución económica del turismo. Estos datos provienen de la hoja "PIB Alojamiento & Serv. comidas" del archivo fuente.

La función `limpiar_pib` maneja la limpieza y estandarización de los años y trimestres (incluyendo formatos numéricos y romanos), asegurando que los datos estén en el rango temporal correcto y las métricas sean numéricas para el análisis.

In [8]:
# PIB
import pandas as pd

# Mapeo de trimestres en números romanos a enteros
trimestres_map = {"I": 1, "II": 2, "III": 3, "IV": 4}

def limpiar_pib(header_row=7, usecols="A:I", anio_inicio=2015, anio_fin=2024):
    # Cargar la hoja del PIB respetando el nombre exacto de la pestaña
    df = pd.read_excel(
        ruta_excel,
        sheet_name="PIB Alojamiento & Serv. comidas",
        header=header_row,
        usecols=usecols
    )

    # Filtrar filas donde Año no sea numérico
    df["Año"] = df["Año"].astype(str).str.extract(r"(\d{4})")[0]
    df = df[df["Año"].notna()].copy()
    df["Año"] = df["Año"].astype(int)

    # Filtrar para que esté dentro del rango 2015 - 2024
    df = df[(df["Año"] >= anio_inicio) & (df["Año"] <= anio_fin)].copy()

    # Limpieza de Trimestre
    df["Trimestre_Texto"] = df["Trimestre"].astype(str).str.strip().str.upper()

    mapa_trimestres = {
        "I": 1, "II": 2, "III": 3, "IV": 4,
        "1": 1, "2": 2, "3": 3, "4": 4,
        "1.0": 1, "2.0": 2, "3.0": 3, "4.0": 4
    }
    df["Trimestre"] = df["Trimestre_Texto"].map(mapa_trimestres)

    # Descartar filas que no correspondan a trimestres válidos
    df = df.dropna(subset=["Trimestre"]).copy()
    df["Trimestre"] = df["Trimestre"].astype(int)

    # Eliminar columna auxiliar
    df = df.drop(columns=["Trimestre_Texto"])

    # Limpiar y asegurar que las columnas métricas sean numéricas
    columnas_metricas = [col for col in df.columns if col not in ["Año", "Trimestre"]]

    # Separar columnas de variación
    columnas_var = [c for c in columnas_metricas if c.strip().lower().startswith("var")]
    columnas_nivel = [c for c in columnas_metricas if c not in columnas_var]

    for col in columnas_metricas:
        if df[col].dtype == object:
            df[col] = (df[col].astype(str)
                              .str.replace("%", "", regex=False)
                              .str.replace(" ", "", regex=False)
                              .str.replace(",", ".", regex=False))
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Ordenar cronológicamente por Año y Trimestre
    df = df.sort_values(["Año", "Trimestre"]).reset_index(drop=True)

    # Redondear solo los niveles; las variaciones conservan decimales
    df[columnas_nivel] = df[columnas_nivel].round(0)
    df[columnas_var] = df[columnas_var].round(2)

    return df

#Cargar la hoja de PIB
df_pib = limpiar_pib(header_row=7, usecols="A:I", anio_inicio=2015, anio_fin=2024)

# Formato de visualización por tipo de columna
col_var = [c for c in df_pib.columns if c.strip().lower().startswith("var")]
col_niv = [c for c in df_pib.columns if c not in col_var + ["Año", "Trimestre"]]

fmt = {c: (lambda x: f"{x:,.0f}".replace(",", ".")) for c in col_niv}
fmt.update({c: "{:,.2f}" for c in col_var})

print(df_pib.shape)
df_pib.head(10).style.format(fmt)


df_pib_alojamientos = df_pib.copy()


(40, 9)


### 5.6. Visitantes Extranjeros por Continente de Origen
Esta sección se enfoca en la desagregación de visitantes extranjeros por su continente de origen, proporcionando una perspectiva geográfica sobre los mercados turísticos. Los datos se cargan tuilizando una API oficial gubernamental desde un archivo CSV externo (datos.gov.co).

La función `limpiar_entradas_datos_gov` realiza la carga del CSV, la limpieza de años, el mapeo de meses a trimestres, la conversión de totales a numéricos y, de manera crucial, el mapeo de la nacionalidad a su continente correspondiente. Finalmente, los datos se agrupan por continente a nivel trimestral.

In [9]:
import pandas as pd

# Mapeo de meses
meses_map = {
    "enero":1, "febrero":2, "marzo":3, "abril":4, "mayo":5, "junio":6,
    "julio":7, "agosto":8, "septiembre":9, "octubre":10, "noviembre":11, "diciembre":12
}

# Diccionario de mapeo de continentes según la nacionalidad registrada
mapa_continentes = {
    # América
    'Estados Unidos': 'América', 'Canadá': 'América', 'México': 'América', 'Brasil': 'América',
    'Argentina': 'América', 'Chile': 'América', 'Perú': 'América', 'Ecuador': 'América',
    'Venezuela': 'América', 'Panamá': 'América', 'Costa Rica': 'América', 'Guatemala': 'América',
    'Uruguay': 'América', 'Paraguay': 'América', 'Bolivia': 'América', 'República Dominicana': 'América',
    'Cuba': 'América', 'Honduras': 'América', 'El Salvador': 'América', 'Nicaragua': 'América',
    'Puerto Rico': 'América', 'Jamaica': 'América', 'Haití': 'América', 'Antigua y Barbuda': 'América',
    'Bahamas': 'América', 'Barbados': 'América', 'Belice': 'América', 'Dominica': 'América',
    'Granada': 'América', 'Guyana': 'América', 'San Cristóbal y Nieves': 'América',
    'Santa Lucía': 'América', 'San Vicente y las Granadinas': 'América', 'Surinam': 'América',
    'Trinidad y Tobago': 'América', 'Aruba': 'América', 'Curaçao': 'América',

    # Europa
    'España': 'Europa', 'Alemania': 'Europa', 'Francia': 'Europa', 'Italia': 'Europa',
    'Reino Unido': 'Europa', 'Países Bajos': 'Europa', 'Suiza': 'Europa', 'Bélgica': 'Europa',
    'Portugal': 'Europa', 'Polonia': 'Europa', 'Rusia': 'Europa', 'Suecia': 'Europa',
    'Noruega': 'Europa', 'Dinamarca': 'Europa', 'Finlandia': 'Europa', 'Irlanda': 'Europa',
    'Austria': 'Europa', 'Grecia': 'Europa', 'República Checa': 'Europa', 'Hungría': 'Europa',
    'Rumanía': 'Europa', 'Ucrania': 'Europa', 'Albania': 'Europa', 'Andorra': 'Europa',
    'Bielorrusia': 'Europa', 'Bosnia y Herzegovina': 'Europa', 'Bulgaria': 'Europa',
    'Croacia': 'Europa', 'Chipre': 'Europa', 'Eslovaquia': 'Europa', 'Eslovenia': 'Europa',
    'Estonia': 'Europa', 'Islandia': 'Europa', 'Letonia': 'Europa', 'Lituania': 'Europa',
    'Luxemburgo': 'Europa', 'Malta': 'Europa', 'Mondavia': 'Europa', 'Mónaco': 'Europa',
    'Montenegro': 'Europa', 'Macedonia del Norte': 'Europa', 'San Marino': 'Europa',
    'Serbia': 'Europa', 'Vaticano': 'Europa',

    # Asia
    'China': 'Asia', 'Japón': 'Asia', 'Corea del Sur': 'Asia', 'India': 'Asia',
    'Israel': 'Asia', 'Turquía': 'Asia', 'Filipinas': 'Asia', 'Indonesia': 'Asia',
    'Malasia': 'Asia', 'Singapur': 'Asia', 'Tailandia': 'Asia', 'Vietnam': 'Asia',
    'Emiratos Árabes Unidos': 'Asia', 'Arabia Saudita': 'Asia', 'Pakistán': 'Asia',
    'Irán': 'Asia', 'Irak': 'Asia', 'Catar': 'Asia', 'Kuwait': 'Asia', 'Líbano': 'Asia',

    # África
    'Sudáfrica': 'África', 'Egipto': 'África', 'Marruecos': 'África', 'Nigeria': 'África',
    'Kenia': 'África', 'Túnez': 'África', 'Argelia': 'África', 'Ghana': 'África',
    'Costa de Marfil': 'África', 'Senegal': 'África',

    # Oceanía
    'Australia': 'Oceanía', 'Nueva Zelanda': 'Oceanía', 'Fiyi': 'Oceanía',
    'Papúa Nueva Guinea': 'Oceanía', 'Samoa': 'Oceanía'
}

def limpiar_entradas_datos_gov(url_csv, anio_inicio=2015, anio_fin=2024):
    # Cargar CSV desde la URL
    df = pd.read_csv(url_csv)

    # Limpiar espacios en los nombres de columnas
    df.columns = df.columns.astype(str).str.strip()

    # Asegurar que Año sea entero y filtrar desde 2015 en adelante
    df["Año"] = pd.to_numeric(df["Año"], errors="coerce")
    df = df.dropna(subset=["Año"]).copy()
    df["Año"] = df["Año"].astype(int)
    df = df[(df["Año"] >= anio_inicio) & (df["Año"] <= anio_fin)].copy()

    # Mapear Mes a Trimestre
    df["Mes_Texto"] = df["Mes"].astype(str).str.strip().str.lower()
    df["mes_num"] = df["Mes_Texto"].map(meses_map)
    df["Trimestre"] = ((df["mes_num"] - 1) // 3) + 1

    df = df.dropna(subset=["Trimestre"]).copy()
    df["Trimestre"] = df["Trimestre"].astype(int)
    df["Total"] = pd.to_numeric(df["Total"], errors="coerce").fillna(0)

    # 5. Mapear Nacionalidad a Continente
    df["Nacionalidad"] = df["Nacionalidad"].astype(str).str.strip()
    df["Continente"] = df["Nacionalidad"].map(mapa_continentes).fillna("Otros")

    # 6. Agrupar por Año, Trimestre y Continente
    df_continentes = (
        df.groupby(["Año", "Trimestre", "Continente"])["Total"]
        .sum()
        .unstack("Continente")
        .add_prefix("Continente_")
        .fillna(0)
        .reset_index()
    )
    df_continentes.columns.name = None

    # Ordenar cronológicamente
    df_continentes = df_continentes.sort_values(["Año", "Trimestre"]).reset_index(drop=True)

    return df_continentes

#Ejecución
URL0 = "https://www.datos.gov.co/api/v3/views/96sh-4v8d/export.csv?cacheBust=1787261427&accessType=DOWNLOAD"
df_continentes = limpiar_entradas_datos_gov(URL0, anio_inicio=2015, anio_fin=2024)

# Visualización
print(df_continentes.shape)

cols_num = [col for col in df_continentes.columns if col not in ["Año", "Trimestre"]]

df_continentes.head(10).style.format({
    col: lambda x: f"{x:,.0f}".replace(',', '.') if pd.notna(x) else '0'
    for col in cols_num
})

Visitantes_extranjeros_por_continente_Origen = df_continentes.copy()

(40, 8)


### 6.1. Verificación de Columnas Clave y Tipos de Datos
Se genera un DataFrame de diagnóstico para inspeccionar si todas las tablas contienen las columnas `Año` y `Trimestre`, así como sus respectivos tipos de datos (`dtype`) y los valores únicos presentes. Esto es crucial para identificar cualquier inconsistencia antes de intentar unir los datos.

### La copia de las tablas se llaman asi:

En caso de borrar algun dato accidentalmente, se realizaron copias para los dataframes.


df_sexo_motivo_ciudad,
df_poblacion_ocupada,
df_pais_residencia,
Trm_Trimestral,
df_visitantes_no_residentes,
df_pib_alojamientos,
Visitantes_extranjeros_por_continente_Origen,

In [10]:
#Diagnotico de Compatibilidad  pre unión para verificar filas repetidas o con nombres iguales
import pandas as pd
# Diccionario con todas tus tablas
tablas = {
    "df_sexo_motivo_ciudad": df_sexo_motivo_ciudad,
    "df_poblacion_ocupada": df_poblacion_ocupada,
    "Trm_Trimestral": Trm_Trimestral,
    "df_visitantes_no_residentes": df_visitantes_no_residentes,
    "df_pib_alojamientos": df_pib_alojamientos,
    "Visitantes_extranjeros_por_continente_Origen": Visitantes_extranjeros_por_continente_Origen
}

col_anio = "Año"
col_trim = "Trimestre"

diagnostico = []

for nombre, df in tablas.items():
    fila = {"tabla": nombre}

    # 1. Verificar que las columnas existan
    cols_reales = df.columns.tolist()
    fila["tiene_col_anio"] = col_anio in cols_reales
    fila["tiene_col_trim"] = col_trim in cols_reales

    if fila["tiene_col_anio"]:
        fila["dtype_anio"] = df[col_anio].dtype
        fila["valores_unicos_anio"] = sorted(df[col_anio].dropna().unique().tolist())

    if fila["tiene_col_trim"]:
        fila["dtype_trim"] = df[col_trim].dtype
        fila["valores_unicos_trim"] = sorted(df[col_trim].dropna().unique().tolist(), key=str)

    diagnostico.append(fila)

diagnostico_df = pd.DataFrame(diagnostico)
diagnostico_df

,tabla,tiene_col_anio,tiene_col_trim,dtype_anio,valores_unicos_anio,dtype_trim,valores_unicos_trim
0,df_sexo_motivo_ciudad,True,True,int64,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",int64,"[1, 2, 3, 4]"
1,df_poblacion_ocupada,True,True,int64,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",int64,"[1, 2, 3, 4]"
2,Trm_Trimestral,True,True,int64,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",int64,"[1, 2, 3, 4]"
3,df_visitantes_no_residentes,True,True,int64,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",int64,"[1, 2, 3, 4]"
4,df_pib_alojamientos,True,True,int64,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",int64,"[1, 2, 3, 4]"
5,Visitantes_extranjeros_por_continente_Origen,True,True,int64,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",int64,"[1, 2, 3, 4]"


### 6.2. Verificación del Rango de Años Común
Para asegurar que la unión sea efectiva y que la base maestra tenga un rango temporal consistente, se verifica que todas las tablas abarquen el mismo periodo de años, que se ha definido entre **2015 y 2024**.

In [11]:
for nombre, df in tablas.items():
    anios = sorted(df["Año"].dropna().unique())
    print(f"{nombre}: {anios[0]} - {anios[-1]}  (n={len(anios)} años)")

df_sexo_motivo_ciudad: 2015 - 2024  (n=10 años)
df_poblacion_ocupada: 2015 - 2024  (n=10 años)
Trm_Trimestral: 2015 - 2024  (n=10 años)
df_visitantes_no_residentes: 2015 - 2024  (n=10 años)
df_pib_alojamientos: 2015 - 2024  (n=10 años)
Visitantes_extranjeros_por_continente_Origen: 2015 - 2024  (n=10 años)


## 7. Creación de la Base de Datos Maestra
Una vez confirmada la compatibilidad de todas las tablas procesadas, se procede a unirlas en una única base de datos maestra (`tabla_maestra`). Esta integración se realiza utilizando la función `reduce` de Python en conjunto con `pd.merge`, lo que permite aplicar una serie de uniones sucesivas entre múltiples DataFrames.

El método de unión elegido es `how="outer"`. Esto significa que todas las filas de todas las tablas se mantendrán en la base maestra. Si un `Año` y `Trimestre` particular existe en una tabla pero no en otra, los valores de las columnas de la tabla faltante se rellenarán con `NaN` (Not a Number), garantizando que no se pierda información y que la base maestra contenga un registro completo de todos los trimestres presentes en las fuentes.

Finalmente, la base maestra se ordena cronológicamente por `Año` y `Trimestre` para facilitar cualquier análisis posterior.

In [12]:
# Importamos "reduce" de la librería functools.
# La necesitamos porque vamos a aplicar un merge (una unión) no entre 2 tablas,
# sino entre 6. reduce nos deja aplicar una operación en cadena:
# une la tabla 1 con la 2, el resultado lo une con la 3, ese resultado con la 4, y así.
from functools import reduce

# Aquí metemos las 6 tablas que ya confirmamos que comparten
# el mismo rango de años (2015-2024) y el mismo formato en Trimestre (1,2,3,4).
# El orden en la lista no afecta el resultado, el merge las va a cruzar todas igual.
tablas_lista = [
    df_sexo_motivo_ciudad,
    df_poblacion_ocupada,
    Trm_Trimestral,
    df_visitantes_no_residentes,
    df_pib_alojamientos,
    Visitantes_extranjeros_por_continente_Origen,
]

# Esta es la parte central. pd.merge junta dos tablas comparando
# los valores de las columnas que le indiquemos (en este caso "Año" y "Trimestre").
# Si una fila tiene Año=2018 y Trimestre=2 en ambas tablas, esa fila se junta en una sola línea.
#
# El parámetro how="outer" dice: no botes ninguna fila aunque no tenga pareja en la otra tabla.
# Si un Año-Trimestre existe en una tabla pero no en otra, igual se conserva la fila,
# y donde falte el dato se rellena con NaN (vacío) en vez de perder esa fila.
#
# reduce va aplicando este merge de a dos tablas a la vez, en cadena,
# hasta que las 6 quedan unidas en una sola.
tabla_maestra = reduce(
    lambda izq, der: pd.merge(izq, der, on=["Año", "Trimestre"], how="outer"),
    tablas_lista
)

# Ordenamos el resultado final por Año y luego por Trimestre,
# para que quede en orden cronológico y no como salió del merge (que puede quedar desordenado).
# reset_index(drop=True) reacomoda el número de fila (el índice) de 0 en adelante,
# porque después de ordenar, los números de índice quedan desordenados y eso es solo cosmético.
tabla_maestra = tabla_maestra.sort_values(["Año", "Trimestre"]).reset_index(drop=True)
cols_porcentaje = [col for col in tabla_maestra.columns if "Var." in col or "%" in col]
cols_montos = [col for col in tabla_maestra.columns if col not in ["Año", "Trimestre"] + cols_porcentaje]

formato_maestro = {col: lambda x: f"{x:,.0f}".replace(',', '.') if pd.notna(x) else '' for col in cols_montos}
formato_maestro.update({col: lambda x: f"{x:,.2f}".replace('.', ',') if pd.notna(x) else '' for col in cols_porcentaje})
# Mostramos el resultado para revisarlo
tabla_maestra.style.format(formato_maestro)

,Año,Trimestre,region_Amazonía,region_Andina,region_Caribe,region_Insular,region_Orinoquía,region_Pacífica,region_Sin Especificar,motivo_Educación y formación,motivo_Negocios y motivos profesionales,motivo_Otros motivos,motivo_Religión y peregrinaciones,motivo_Salud y atención médica,motivo_Tránsito,"motivo_Vacaciones, recreo y ocio",motivo_Visitas a familiares y amigos,sexo_Femenino,sexo_Masculino,sexo_Otro,Ocupados Total Nacional,Comercio y reparación de vehículos,Alojamiento y servicios de comida_x,Transporte y almacenamiento,TRM_promedio,TRM_cierre,Extranjeros no residentes sin venezolanos,Residentes en Venezuela por motivo de Descanso y Esparcimiento,Colombianos residentes exterior,Pasajeros en cruceros internacionales,Total visitantes no residentes,Comercio al por mayor y por menor*,Transporte **,Alojamiento y servicios de comida_y,Subtotal Valor Agregado,PIB,Var. PIB respecto al mismo período del año anterior,Var. Aloj&Serv. Comida respecto al mismo período del año anterior.,Continente_América,Continente_Asia,Continente_Europa,Continente_Oceanía,Continente_Otros,Continente_África
0,2015,1,1.268,300.140,98.743,30.058,992,38.295,458,9.312,93.825,35.568,629,2.471,714,327.426,9,185.251,284.703,,20.290,3.742,1.498,1.444,2.475,2.383,469.954,0,154.823,99.537,724.314,13.454,7.858,7.022,173.725,191.362,"0,03","0,02",70.303,11.278,33.087,3.724,27.437,689
1,2015,2,1.733,309.320,76.282,20.273,958,42.768,119,11.486,100.890,35.113,374,2.429,2.046,299.110,5,174.000,277.453,,20.881,3.869,1.463,1.497,2.496,2.598,451.453,0,177.883,58.727,688.063,14.120,8.004,7.225,178.673,196.861,"0,03","0,04",57.096,9.529,29.374,3.679,25.763,712
2,2015,3,1.951,359.632,83.037,20.687,1.095,54.685,105,16.160,102.420,39.118,354,2.494,1.934,358.712,,207.083,314.109,,20.838,3.893,1.521,1.447,2.936,2.599,521.192,0,191.080,11.000,723.272,14.526,8.316,7.689,185.452,203.636,"0,04","0,05",58.201,9.836,33.600,4.088,24.534,1.033
3,2015,4,1.208,363.820,97.489,20.112,1.427,58.024,65,12.260,91.859,38.771,309,2.294,2.082,394.570,,213.014,329.131,,21.431,4.027,1.472,1.387,3.063,3.087,542.145,0,284.180,102.942,929.267,17.299,8.568,7.925,192.693,212.833,"0,02","0,06",61.281,11.057,34.718,3.971,22.634,675
4,2016,1,1.364,358.000,128.195,29.878,1.137,47.960,26,10.549,88.458,39.496,479,2.805,623,424.149,1,233.381,333.179,,20.408,3.861,1.507,1.502,3.262,3.213,566.560,0,171.546,128.080,866.186,13.925,7.976,7.539,178.014,195.907,"0,02","0,07",68.392,10.421,38.731,4.452,26.855,846
5,2016,2,1.472,344.798,95.798,22.934,1.049,48.394,49,12.146,102.175,35.613,305,3.433,522,360.300,,203.865,310.629,,21.007,3.877,1.538,1.439,2.993,3.001,514.494,0,189.659,45.302,749.455,14.601,7.952,7.549,182.852,201.039,"0,02","0,04",56.231,9.254,32.275,4.270,22.218,758
6,2016,3,1.706,387.868,106.185,25.129,1.326,57.437,140,16.843,100.543,39.494,353,3.340,1.142,418.076,,239.249,340.542,,20.844,3.935,1.486,1.448,2.949,2.919,579.791,0,197.158,8.743,785.692,14.972,8.291,8.025,188.001,206.806,"0,02","0,04",60.500,11.117,35.145,4.546,23.837,830
7,2016,4,1.521,387.932,106.019,23.431,1.456,59.824,26,14.509,92.949,37.975,284,3.300,583,430.609,,234.635,345.574,,21.464,4.065,1.557,1.448,3.014,2.880,580.209,0,281.966,123.695,985.870,18.140,8.634,7.952,197.666,217.736,"0,02","0,00",65.736,11.405,36.472,4.175,20.550,937
8,2017,1,1.953,369.989,137.175,34.810,1.304,48.469,50,10.585,95.963,38.260,485,3.645,682,444.129,1,246.139,347.611,,20.590,3.983,1.430,1.490,2.922,2.981,593.750,0,172.320,140.485,906.555,14.141,8.058,7.595,180.126,198.002,"0,01","0,01",68.781,11.526,41.167,3.628,23.048,748
9,2017,2,2.500,377.113,116.352,25.283,1.212,52.528,11,13.192,98.868,37.616,309,3.770,324,420.920,,238.180,336.819,,21.310,3.892,1.467,1.475,2.920,2.886,574.999,0,201.942,60.056,836.997,14.914,8.157,7.692,185.271,203.591,"0,01","0,02",60.327,10.325,33.324,4.487,20.177,742
